# Dataset

Lo studio del dataset costruito (`full_v2-built`, lo stesso pubblicato su Hugging Face): è esattamente quello che vedrà il
modello. Un'app alla volta: cambia `APP` nella cella dei parametri e rilancia tutto.

**Una run.** `x [T, S, F]` è il valore di ogni feature per passo di 30 s, servizio e feature (NaN dove nel passo non è
caduto nessun campione). Il carico di wrk2 parte a `t_workload`; dopo 300 s di settle il guasto è iniettato a
`t_inject`; poi 900 s di osservazione (5400 s nelle sessioni pulite, che non iniettano nulla). `t_rel` sono i secondi
dall'iniezione: negativi nel settle.

**Split.** `clean`: 12 sessioni sane da 90 minuti, il training. `dev`: guasti diversi da quelli del test, per il tuning.
`test`: 5 run per scenario, ognuna con il guasto iniettato a un livello di carico diverso (10, 20, 40, 80, 160 req/s).

Le **Letture** sono scritte sui numeri di `full_v2-built` costruito con il builder 1533dfa, per entrambe le app.

In [ ]:
import json
import sys
import warnings
from pathlib import Path
from types import SimpleNamespace

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "anomaly" / "build.py").exists())
sys.path.insert(0, str(REPO))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anomaly import explore as ex

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_colwidth", 90, "display.max_rows", 80)

# oppure: from huggingface_hub import snapshot_download
#         ROOT = Path(snapshot_download("Keeevin16/AIOpsLab-app-collection-Hotel-Social", repo_type="dataset",
#                                       allow_patterns=["*/*/*.npz", "*/*/*.json"]))
ROOT = Path.home() / "datasets" / "full_v2-built"
APP = "hotel"       # "hotel" o "social"
RUN = None          # run della sezione 1, per esempio "test/network_delay_hotel_res__r1"; None = la prima del test
SPLIT = "dev"       # run di guasto della sezione 6: "dev" o "test"
MEM_DELTA = False   # True: nelle sezioni 4-5 la memoria diventa la variazione per passo (MiB) invece del livello

## 0 · Caricamento

- `runs`: le run dell'app, per chiave `split/nome`. Ogni run ha gli stessi campi di `explore.load`, così gli helper di
  `explore` (`robust_z`, `settle_mask`, `steady_mask`, `target_services`, `score_services`) funzionano anche qui.
- `info`: una riga per run.
- `df`: tutta l'app in formato lungo, una riga per (run, passo, servizio) e una colonna per feature, più `rate`, `mix`,
  `split`, `phase` (`off` fuori dal carico, `before` fino all'iniezione, `after` dopo) e `steady` (carico fermo da almeno
  60 s). È la tabella su cui lavorare con `groupby` e `query`.
- `prep(run)`: le feature come le vedrebbe il modello: CPU in millicore, memoria in MiB, restart ogni 5 minuti, poi
  `log1p`. `Z` sono gli stessi valori standardizzati con media e deviazione delle sessioni pulite, per servizio e feature.

In [ ]:
def load_built(npz):
    z, labels = np.load(npz), json.load(open(npz.with_suffix(".json")))
    run = SimpleNamespace(**{k: z[k] for k in z.files}, labels=labels)
    run.services, run.features = [str(s) for s in run.services], [str(f) for f in run.features]
    run.split, run.id, run.kind = npz.parent.name, f"{npz.parent.name}/{npz.stem}", labels["kind"]
    run.t_rel = run.t - labels["t_inject"]
    return run


def scenario(r):
    return r.id.split("/")[1].rsplit("__", 1)[0].removesuffix("_hotel_res").removesuffix("_social_network")


def rate_at_injection(r):
    return int(r.rate[min(np.searchsorted(r.t, r.labels["t_inject"]), len(r.t) - 1)])


runs = {r.id: r for r in map(load_built, sorted((ROOT / APP).glob("*/*.npz")))}
first = next(iter(runs.values()))
SERVICES, FEATURES, STEP = first.services, first.features, first.labels["step"]
assert all(r.services == SERVICES and r.features == FEATURES for r in runs.values())
clean_ids = [k for k, r in runs.items() if r.split == "clean"]

info = pd.DataFrame([{
    "run": r.id, "split": r.split, "scenario": scenario(r), "rep": r.labels["rep"], "fault": r.labels["fault_type"],
    "target": ", ".join(ex.target_services(r)) or "-", "rate_inj": rate_at_injection(r), "steps": len(r.t),
    "minutes": round((r.t[-1] - r.t[0]) / 60),
} for r in runs.values()]).set_index("run")


def frame(runs):
    parts = []
    for r in runs.values():
        T, S, F = r.x.shape
        index = pd.MultiIndex.from_arrays([np.repeat(r.id, T * S), np.repeat(r.t_rel, S), np.tile(r.services, T)],
                                          names=["run", "t_rel", "service"])
        part = pd.DataFrame(r.x.reshape(T * S, F).astype(np.float64), index=index, columns=r.features)
        phase = np.where(r.rate < 0, "off", np.where(r.t_rel <= 0, "before", "after"))
        part = part.assign(rate=np.repeat(r.rate, S), mix=np.repeat(r.mix, S), split=r.split,
                           phase=np.repeat(phase, S), steady=np.repeat(ex.steady_mask(r), S))
        parts.append(part)
    return pd.concat(parts)


df = frame(runs)
clean = df[(df.split == "clean") & (df.rate >= 0)]
steady_clean = clean[clean.steady]

UNIT = {"cpu": 1e3, "cpu_user": 1e3, "cpu_system": 1e3, "cpu_throttled": 1e3,  # millicore
        "mem_working_set": 2**-20, "mem_rss": 2**-20, "mem_cache": 2**-20,     # MiB
        "restarts": 300}                                                          # per 5 minuti
MEMORY = [f for f in FEATURES if f.startswith("mem_")]


def prep(r, mem_delta=MEM_DELTA):
    v = np.clip(r.x, 0, None) * np.array([UNIT.get(f, 1) for f in FEATURES])
    out = np.log1p(v)
    if mem_delta:
        for f in MEMORY:
            j = FEATURES.index(f)
            d = np.diff(v[:, :, j], axis=0, prepend=np.nan)
            out[:, :, j] = np.sign(d) * np.log1p(np.abs(d))
    return out


P = {k: prep(r) for k, r in runs.items()}
ref = np.concatenate([P[k][runs[k].rate >= 0] for k in clean_ids])
MU, SD = np.nanmean(ref, axis=0), np.nanstd(ref, axis=0)
Z = {k: (v - MU) / np.where(SD > 1e-6, SD, np.nan) for k, v in P.items()}


def heat(data, title, fmt="{:.2f}", cmap="RdBu_r", vmin=-1, vmax=1, annot=True, ax=None):
    # annotated heatmap of a DataFrame: rows = index, columns = columns
    if ax is None:
        _, ax = plt.subplots(figsize=(1.5 + .6 * data.shape[1], 1 + .3 * data.shape[0]))
    a = data.to_numpy(dtype=float)
    ax.imshow(a, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    ax.set_xticks(range(data.shape[1]), [str(c) for c in data.columns], rotation=60, ha="right", fontsize=7)
    ax.set_yticks(range(data.shape[0]), [str(i) for i in data.index], fontsize=7)
    if annot:
        mid, half = (vmin + vmax) / 2, (vmax - vmin) / 2
        for (i, k), v in np.ndenumerate(a):
            if not np.isnan(v):
                ax.text(k, i, fmt.format(v), ha="center", va="center", fontsize=6,
                        color="white" if abs(v - mid) > .6 * half and cmap != "Reds" or cmap == "Reds" and v > mid + .6 * half else "black")
    ax.set_title(title, fontsize=10)
    ax.figure.tight_layout()
    return ax


print(f"{APP}: {len(runs)} run, {len(SERVICES)} servizi, {len(FEATURES)} feature, df {df.shape}")
info.groupby(["split", "scenario"]).agg(run=("rep", "size"), rate_inj=("rate_inj", lambda v: sorted(v)),
                                        minuti=("minutes", "median"), target=("target", "first"))

## 1 · Una run

*Cosa succede in una run, dall'inizio del carico alla fine.* Sopra il carico di wrk2 (rate e mix), con l'iniezione a 0.
Sotto un pannello per feature: righe = servizi, colonne = tempo, colore = z robusto rispetto al settle
(`(x − mediana) / IQR`, tagliato a ±8), grigio = nessun campione. La tabella è ogni segmento di wrk2: rate erogato,
p99 e risposte non 2xx.

Il riferimento è il settle: se dopo il settle il carico cambia, le feature che seguono il carico si colorano anche senza
guasto. È il confondente che il modello deve imparare a ignorare.

In [ ]:
r = runs[RUN] if RUN else next(v for k, v in runs.items() if k.startswith("test/"))
minutes = r.t_rel / 60
print(r.id, "|", r.labels["fault_type"], "| target:", ", ".join(ex.target_services(r)) or "-",
      "| rate all'iniezione:", rate_at_injection(r))

fig, ax = plt.subplots(figsize=(13, 2.2))
ax.step(minutes, np.where(r.rate >= 0, r.rate, np.nan), where="pre")
ax.set_ylabel("req/s")
ax2 = ax.twinx()
ax2.step(minutes, np.where(r.mix >= 0, r.mix, np.nan), where="pre", color="tab:orange", alpha=.6)
ax2.set_ylabel("mix", color="tab:orange")
ax.axvline(0, color="red", ls="--")
ax.set_xlabel("minuti dall'iniezione")
plt.show()

z = np.clip(ex.robust_z(r.x, ex.settle_mask(r)), -8, 8)
cmap = plt.get_cmap("RdBu_r").copy()
cmap.set_bad("#bbbbbb")
fig, axes = plt.subplots(4, 4, figsize=(16, 3 + .5 * len(SERVICES)), sharex=True, sharey=True)
for j, ax in enumerate(axes.flat):
    if j >= len(FEATURES):
        ax.axis("off")
        continue
    ax.imshow(z[:, :, j].T, aspect="auto", cmap=cmap, vmin=-8, vmax=8, interpolation="nearest",
              extent=[minutes[0] - STEP / 60, minutes[-1], len(SERVICES) - .5, -.5])
    ax.axvline(0, color="k", lw=.8)
    ax.set_title(FEATURES[j], fontsize=9)
axes[0, 0].set_yticks(range(len(SERVICES)), SERVICES)
for ax in axes[:, 0]:
    ax.tick_params(axis="y", labelsize=6)
plt.tight_layout()
plt.show()

pd.DataFrame(r.labels["segments"]).assign(start=lambda d: ((d.start - r.labels["t_inject"]) / 60).round(1)) \
    .rename(columns={"start": "inizio (min)"})

## 2 · Qualità

*Quali celle servizio×feature contengono informazione?* Sulle sessioni pulite sotto carico ogni cella è:

- `assente`: nessun campione (il throttling su Social, che non ha limiti di CPU; la cache di servizi senza disco);
- `piatta`: c'è, ma tra il 5° e il 95° percentile si muove meno del minimo cambiamento che conta (`explore.MIN_SCALE`:
  2 mC, 1 MiB, 2 pacchetti/s...). Il modello non impara niente lì, e uno z-score con deviazione quasi zero esplode;
- `viva`: tutto il resto.

In [ ]:
present = clean[FEATURES].notna().groupby(level="service").mean().reindex(SERVICES)
spread = (clean.groupby(level="service")[FEATURES].quantile(.95)
          - clean.groupby(level="service")[FEATURES].quantile(.05)).reindex(SERVICES)
floor = pd.Series(ex.MIN_SCALE)[FEATURES]
status = pd.DataFrame("viva", index=SERVICES, columns=FEATURES)
status = status.mask(spread.lt(floor, axis=1) | spread.isna(), "piatta").mask(present == 0, "assente")
alive = status != "assente"

codes = status.map({"assente": 0, "piatta": 1, "viva": 2}.get)
from matplotlib.colors import ListedColormap
ax = heat(codes, "stato delle celle nelle sessioni pulite: grigio assente, giallo piatta, verde viva",
          cmap=ListedColormap(["#cccccc", "#f2d16b", "#5bb974"]), vmin=0, vmax=2, annot=False)
plt.show()
display(status.apply(lambda c: c.value_counts()).reindex(["viva", "piatta", "assente"]).fillna(0).astype(int))
print("servizi senza nessuna cella viva:", [s for s in SERVICES if not (status.loc[s] == "viva").any()])

*Quanti dati mancano, per split e fase?* Quota di passi senza campione, contata solo sulle celle che esistono.

In [ ]:
keep = alive.loc[df.index.get_level_values("service")].to_numpy()
na = np.array(df[FEATURES].isna(), dtype=float)
na[~keep] = np.nan
missing = pd.DataFrame(na, columns=FEATURES).assign(split=df.split.to_numpy(), phase=df.phase.to_numpy())
missing = missing[missing.phase != "off"].groupby(["split", "phase"]).mean()
missing.insert(0, "media", missing.mean(axis=1))
missing.round(3)

*Che forma hanno le distribuzioni?* Per ogni feature, sulle celle vive delle sessioni pulite: quante decadi separano la
mediana del servizio più piccolo da quella del più grande, e l'asimmetria (skew) mediana sui servizi, grezza e dopo
`prep`. Sotto, i valori di quattro feature per servizio in scala logaritmica.

In [ ]:
def skew(v):
    v = v[~np.isnan(v)]
    s = v.std() if len(v) else 0
    return np.nan if len(v) < 20 or s == 0 else float(((v - v.mean()) ** 3).mean() / s**3)


Xc = np.concatenate([runs[k].x[runs[k].rate >= 0] for k in clean_ids])
Pc = np.concatenate([P[k][runs[k].rate >= 0] for k in clean_ids])
rows = []
for j, f in enumerate(FEATURES):
    live = [s for s in range(len(SERVICES)) if status.iloc[s, j] == "viva"]
    if not live:
        continue
    meds = np.nanmedian(Xc[:, live, j], axis=0)
    pos = meds[meds > 0]
    rows.append({"feature": f, "celle vive": len(live), "decadi tra servizi": np.log10(pos.max() / pos.min()),
                 "skew grezza": np.nanmedian([skew(Xc[:, s, j]) for s in live]),
                 "skew dopo prep": np.nanmedian([skew(Pc[:, s, j]) for s in live])})
display(pd.DataFrame(rows).set_index("feature").round(2))

SHOW = ["cpu_user", "mem_working_set", "rx_bytes", "threads"]
fig, axes = plt.subplots(1, len(SHOW), figsize=(16, 1 + .3 * len(SERVICES)), sharey=True)
for ax, f in zip(axes, SHOW):
    j = FEATURES.index(f)
    vals = [Xc[:, s, j][~np.isnan(Xc[:, s, j])] * UNIT.get(f, 1) for s in range(len(SERVICES))]
    ax.boxplot([v[v > 0] if (v > 0).any() else [np.nan] for v in vals], orientation="horizontal", showfliers=False)
    ax.set_xscale("log")
    ax.set_title(f + {"cpu_user": " (mC)", "mem_working_set": " (MiB)", "rx_bytes": " (B/s)"}.get(f, ""), fontsize=9)
axes[0].set_yticks(range(1, len(SERVICES) + 1), SERVICES, fontsize=7)
plt.tight_layout()
plt.show()

**Lettura.** In Hotel CPU e rete sono vive in 12 servizi su 17: i mongo di geo, profile, rate, recommendation e user sono quasi fermi, perché i dati si caricano all'avvio. `cpu_throttled` è piatta in tutti i servizi (nelle sessioni sane il limite di 1000m non si raggiunge) e `mem_cache` è viva in un servizio solo. In Social `cpu_throttled` è assente; media-frontend, media-memcached e url-shorten-memcached non hanno nessuna cella viva, perché il carico non passa di lì; la memoria dei servizi stateless (text, media, unique-id, user-mention, url-shorten) è piatta. `restarts` è piatta ovunque per costruzione (V2: nel training vale sempre 0): va tenuta, perché qualunque restart è anomalo, ma con un minimo sulla scala, altrimenti lo z esplode. Mancano pochi dati: 0.1-0.3% nelle sessioni pulite e prima dell'iniezione, 1.5-2% dopo l'iniezione nel test, dove i guasti spengono i pod. Lì il buco è segnale: serve la maschera, non l'imputazione. La rete copre 2-4 decadi tra servizi (4.2 per `rx_bytes` in Social) con skew 1.4-1.6; dopo `prep` CPU e rete scendono sotto 0.5 in valore assoluto. Memoria e threads non ne hanno bisogno (skew grezza tra -1.2 e 0.9) e con `log1p` diventano asimmetriche a sinistra: il logaritmo serve solo a CPU e rete.

## 3 · Carico

*Quali feature seguono il carico?* Correlazione di Spearman tra ogni feature e il rate di wrk2, per servizio, sui passi a
carico fermo delle sessioni pulite. Vicino a 1: la feature si muove con il traffico, e il modello deve imparare quella
relazione per non scambiare un aumento di carico per un guasto. Vicino a 0: indipendente dal carico.

In [ ]:
def spearman(a, b):
    ok = a.notna() & b.notna()
    return a[ok].rank().corr(b[ok].rank()) if ok.sum() > 20 and a[ok].nunique() > 1 else np.nan


rho = steady_clean.groupby(level="service").apply(
    lambda d: pd.Series({f: spearman(d[f], d.rate) for f in FEATURES})).reindex(SERVICES)
rho = rho.where(status == "viva")
heat(rho, "Spearman(feature, rate), celle vive, sessioni pulite a carico fermo")
plt.show()
rho.median().round(2).to_frame("mediana sui servizi").T

*Come risponde una feature al carico e quanto conta il mix?* Mediana di `FEATURE_LOAD` per livello di rate (asse x) e
mix (una linea per mix), un pannello per servizio, scala log-log. Linee parallele = il mix sposta solo il livello;
linee che si incrociano = il mix cambia il lavoro del servizio. Hotel: mix 1 = più ricerche, 2 = più raccomandazioni,
3 = più prenotazioni. Social: 1 e 2 = più letture di timeline.

In [ ]:
FEATURE_LOAD = "cpu_user"
lev = steady_clean.groupby([steady_clean.index.get_level_values("service"), "mix", "rate"])[FEATURE_LOAD].median()
live = [s for s in SERVICES if status.loc[s, FEATURE_LOAD] == "viva"]
ncol = 6
fig, axes = plt.subplots(-(-len(live) // ncol), ncol, figsize=(16, 2.3 * -(-len(live) // ncol)), squeeze=False)
for ax, s in zip(axes.flat, live):
    for m, d in lev.loc[s].groupby(level="mix"):
        d = d.droplevel("mix")
        ax.plot(d.index, d.to_numpy() * UNIT.get(FEATURE_LOAD, 1), "o-", ms=3, label=f"mix {m}")
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_title(s, fontsize=8)
    ax.tick_params(labelsize=6)
for ax in axes.flat[len(live):]:
    ax.axis("off")
axes[0, 0].legend(fontsize=6)
plt.tight_layout()
plt.show()

**Lettura.** CPU e rete seguono il carico in quasi tutti i servizi (Spearman mediano 0.93-0.96 in entrambe le app); memoria e threads poco (0.10-0.37). Il confondente di carico vive quindi in 8 feature su 13 che si muovono insieme: il modello può impararlo dalle relazioni tra loro, senza ricevere il carico. Il mix conta in modo diverso per servizio: in Hotel il mix 3 (più prenotazioni) moltiplica da 3 a 10 volte la CPU di `user` e alza mongodb-reservation, mentre il mix 1 abbassa recommendation. Le sessioni pulite coprono ogni mix in ogni fascia di rate (sezione 4), quindi il modello vede anche questo.

## 4 · Training contro test

*Le sessioni pulite coprono quello che il modello vedrà?* Prima il carico: minuti passati in ogni fascia di rate, per
split e mix.

In [ ]:
steps = df.xs(SERVICES[0], level="service")[["rate", "mix", "split", "phase"]]
steps = steps[steps.rate >= 0]
(pd.crosstab(pd.cut(steps.rate, [0, 15, 30, 60, 120, 200, 400]), [steps.split, steps.mix]) * STEP / 60).round().astype(int)

*Quanti passi normali del dev e del test cadono fuori dal range delle sessioni pulite?* Il range di ogni cella è tra
lo 0.5° e il 99.5° percentile delle sessioni pulite, allargato di `MIN_SCALE` (il minimo cambiamento che conta). Un passo normale (prima dell'iniezione, o in una run `noop`) fuori
range è un falso allarme probabile con qualsiasi modello: il training non ha mai visto quel valore. Dopo l'iniezione
nelle run di guasto, invece, uscire dal range è il segnale. Nella tabella la quota media sulle celle vive.

In [ ]:
svc = df.index.get_level_values("service")
lo = clean.groupby(level="service")[FEATURES].quantile(.005).reindex(SERVICES) - floor  # tolleranza: il minimo
hi = clean.groupby(level="service")[FEATURES].quantile(.995).reindex(SERVICES) + floor  # cambiamento che conta
out = np.array(df[FEATURES].lt(lo.loc[svc].to_numpy()) | df[FEATURES].gt(hi.loc[svc].to_numpy()), dtype=float)
out[~keep | df[FEATURES].isna().to_numpy()] = np.nan
fault = info.fault.reindex(df.index.get_level_values("run")).ne("No fault").to_numpy()
out = pd.DataFrame(out, columns=FEATURES, index=df.index).assign(split=df.split.to_numpy(), phase=df.phase.to_numpy(),
                                                                 guasto=fault)
out = out[out.phase != "off"]
summary = out.groupby(["split", "guasto", "phase"])[FEATURES].mean()
display(summary.mean(axis=1).round(3).to_frame("quota fuori range"))

normal = out[(out.split != "clean") & (~out.guasto | (out.phase == "before"))]
risk = normal.groupby(level="service")[FEATURES].mean().reindex(SERVICES)
heat(risk, "passi normali di dev e test fuori dal range delle pulite", vmin=0, vmax=.5, cmap="Reds")
plt.show()
risk.stack().sort_values(ascending=False).head(10).round(3).to_frame("quota")

*Le sessioni differiscono tra loro più di quanto una sessione cambia nel tempo?* Per ogni livello (rate, mix) visto da
almeno 3 sessioni pulite: varianza tra le mediane delle sessioni diviso varianza media dentro la sessione, poi la
mediana sui livelli. Sopra 1 (log10 > 0) ogni sessione ha un suo livello: lo z-score globale lo legge come anomalia.

In [ ]:
sc = steady_clean.reset_index()
g = sc.groupby(["service", "rate", "mix", "run"])[FEATURES]
med, var = g.median(), g.var()
lvl = ["service", "rate", "mix"]
n = med.groupby(level=lvl).size()
ratio = (med.groupby(level=lvl).var() / var.groupby(level=lvl).mean())[n >= 3]
ratio = np.log10(ratio.replace([np.inf, 0], np.nan)).groupby(level="service").median().reindex(SERVICES)
ratio = ratio.where(status == "viva")
heat(ratio, "log10(varianza tra sessioni / dentro la sessione), stesso carico", vmin=-2, vmax=2)
plt.show()
ratio.median().round(2).to_frame("mediana sui servizi").T

*Quali memorie crescono durante la sessione?* Pendenza della memoria (MiB/h) in ogni sessione pulita, mediana e range
sulle sessioni. Nei grafici le tre che crescono di più: una linea per sessione pulita, colorata dal rate medio, e le run
`noop` del test tratteggiate. L'asse x sono i minuti dall'inizio del carico.

In [ ]:
def slope_mib_h(r, s, f="mem_working_set"):
    y = r.x[:, SERVICES.index(s), FEATURES.index(f)] / 2**20
    ok = ~np.isnan(y) & (r.rate >= 0)
    return np.polyfit((r.t[ok] - r.t[ok][0]) / 3600, y[ok], 1)[0] if ok.sum() > 20 else np.nan


slopes = pd.DataFrame({k: {s: slope_mib_h(runs[k], s) for s in SERVICES} for k in clean_ids})
growth = pd.DataFrame({"mediana": slopes.median(axis=1), "min": slopes.min(axis=1), "max": slopes.max(axis=1)})
growth = growth.reindex(growth.mediana.abs().sort_values(ascending=False).index)
display(growth.head(8).round(1))

STORES = list(growth.index[:3])
noop = [k for k in runs if k.startswith("test/") and info.loc[k, "fault"] == "No fault"]
mean_rate = {k: runs[k].rate[runs[k].rate >= 0].mean() for k in clean_ids}
norm = plt.Normalize(min(mean_rate.values()), max(mean_rate.values()))
fig, axes = plt.subplots(1, 3, figsize=(16, 3.2))
for ax, s in zip(axes, STORES):
    for k in clean_ids + noop:
        r = runs[k]
        y = r.x[:, SERVICES.index(s), FEATURES.index("mem_working_set")] / 2**20
        x = (r.t - r.labels["t_workload"]) / 60
        style = dict(color=plt.cm.viridis(norm(mean_rate[k])), lw=1) if k in mean_rate else dict(color="k", ls="--", lw=1)
        ax.plot(x, y, **style)
    ax.set_title(s, fontsize=9)
    ax.set_xlabel("minuti dall'inizio del carico")
axes[0].set_ylabel("MiB")
fig.colorbar(plt.cm.ScalarMappable(norm, "viridis"), ax=axes, label="rate medio della sessione")
plt.show()

*Livello o variazione?* Per le stesse memorie, sulle sessioni pulite dopo il settle: correlazione del livello con il
rate, e della variazione per passo (media mobile su 2 minuti) con il rate e con i pacchetti ricevuti dal servizio stesso.
Se la variazione segue il traffico del servizio e il livello no, il livello dipende dalla storia del carico, che una
finestra di pochi minuti non vede; la variazione invece si spiega con quello che c'è nella finestra (`MEM_DELTA = True`).

In [ ]:
def corr(a, b):
    ok = ~np.isnan(a) & ~np.isnan(b)
    return np.corrcoef(a[ok], b[ok])[0, 1] if ok.sum() > 20 and a[ok].std() > 0 and b[ok].std() > 0 else np.nan


rows = []
for s in STORES:
    lv, d, rt, rx = [], [], [], []
    for k in clean_ids:
        r = runs[k]
        on = (r.t > r.labels["t_workload"] + 300) & (r.rate >= 0)
        y = r.x[:, SERVICES.index(s), FEATURES.index("mem_working_set")] / 2**20
        dy = np.convolve(np.diff(y, prepend=np.nan), np.ones(4) / 4, "same")
        lv += list(y[on]); d += list(dy[on]); rt += list(r.rate[on])
        rx += list(r.x[on, SERVICES.index(s), FEATURES.index("rx_packets")])
    lv, d, rt, rx = map(np.array, (lv, d, rt, rx))
    rows.append({"servizio": s, "livello ~ rate": corr(lv, rt), "variazione ~ rate": corr(d, rt.astype(float)),
                 "variazione ~ rx_packets propri": corr(d, rx)})
pd.DataFrame(rows).set_index("servizio").round(2)

**Lettura.** Il carico è coperto: le sessioni pulite arrivano a 240 req/s, dev e test a 160, con gli stessi mix. Fuori range dopo il settle: 0.1% dei passi nelle sessioni pulite, 0.3% (Hotel) e 0.5% (Social) nel test senza guasto, 6% (Hotel) e 11% (Social) dopo un guasto. La fase `before` è al 2-3% in tutti gli split, anche nelle sessioni pulite: è il primo minuto di carico, non uno spostamento tra split. In Social le celle più a rischio sono i threads di user-service (30%) e social-graph-service (23%). Tra sessioni allo stesso carico la rete è stabile (log10 del rapporto circa -1: dieci volte sotto la variazione dentro la sessione), la memoria no (0.7-1: da 5 a 10 volte sopra); anche la CPU di Social ha un livello per sessione (`cpu_user` 0.63). La memoria di Hotel non cresce (al massimo mongodb-reservation, +5 MiB/h). Quella di Social sì: home-timeline-redis +880 MiB/h, post-storage e url-shorten mongo +380, tanto più quanto è alto il carico della sessione. Il livello non segue il rate (0.10-0.11), la variazione segue i pacchetti dello store stesso (0.87-0.89 per i mongo, 0.42 per redis): in Social la memoria va data come variazione (`MEM_DELTA = True`). In Hotel il problema non c'è e la variazione non aggiunge niente (correlazioni sotto 0.5).

## 5 · Struttura

*Quali feature si ripetono?* Prima le identità esatte, poi la correlazione di Spearman tra feature, per servizio, sulle
sessioni pulite a carico fermo, mediana sui servizi. Una correlazione alta nei dati sani non rende una feature inutile:
è proprio la relazione che il VAE impara, e un guasto la rompe (CPU che sale senza traffico, pacchetti senza CPU).
Toglierla ha senso solo quando è un'identità.

In [ ]:
c = clean[["cpu", "cpu_user", "cpu_system"]].dropna()
print(f"cpu − (cpu_user + cpu_system): mediana |residuo| {(c.cpu - c.cpu_user - c.cpu_system).abs().median():.2e} core,"
      f" cpu mediana {c.cpu.median():.2e} core")
ranks = steady_clean[FEATURES].groupby(level="service").rank()
fcorr = ranks.groupby(level="service").corr().groupby(level=1).median().reindex(index=FEATURES, columns=FEATURES)
heat(fcorr, "Spearman tra feature, mediana sui servizi (sessioni pulite, carico fermo)")
plt.show()

*Quante dimensioni hanno davvero i dati di un servizio?* PCA per servizio sui valori `Z` delle sessioni pulite (celle
vive, senza `DROP`): quante componenti servono per il 90, 95 e 99% della varianza, quanta varianza porta la prima, e
quanto la prima segue il rate. È il punto di partenza per la dimensione latente: un servizio con k95 = 2 non ha bisogno
di un latente da 16 per passo.

In [ ]:
DROP = ["cpu"]  # = cpu_user + cpu_system
rows, curves = [], {}
for s, name in enumerate(SERVICES):
    cols = [j for j, f in enumerate(FEATURES) if f not in DROP and status.loc[name, f] == "viva"]
    if len(cols) < 2:
        continue
    v = np.concatenate([Z[k][runs[k].rate >= 0][:, s, cols] for k in clean_ids])
    rt = np.concatenate([runs[k].rate[runs[k].rate >= 0] for k in clean_ids]).astype(float)
    ok = ~np.isnan(v).any(axis=1)
    v, rt = v[ok] - v[ok].mean(axis=0), rt[ok]
    u, sv, _ = np.linalg.svd(v, full_matrices=False)
    ev = np.cumsum(sv**2) / (sv**2).sum()
    curves[name] = ev
    rows.append({"servizio": name, "feature vive": len(cols), "k90": int(np.searchsorted(ev, .90) + 1),
                 "k95": int(np.searchsorted(ev, .95) + 1), "k99": int(np.searchsorted(ev, .99) + 1),
                 "var PC1": ev[0], "|corr(PC1, rate)|": abs(corr(u[:, 0], rt))})
pca = pd.DataFrame(rows).set_index("servizio")
display(pca.round(2))
fig, ax = plt.subplots(figsize=(7, 3.5))
for name, ev in curves.items():
    ax.plot(range(1, len(ev) + 1), ev, "o-", ms=2, lw=1)
ax.axhline(.95, color="k", ls=":")
ax.set_xlabel("componenti")
ax.set_ylabel("varianza spiegata cumulata")
plt.show()

*Quanto deve essere lunga la finestra W?* Autocorrelazione dei valori `Z` nelle sessioni pulite, per lag da 1 a 20
passi (30 s - 10 min), mediana sulle celle vive e sulle sessioni. A sinistra i valori così come sono: dominati dai
gradini di carico. A destra il residuo dopo aver tolto la mediana di ogni segmento di carico (rate e mix costanti):
la dinamica propria dei segnali. Sotto, la durata dei segmenti di carico (senza i gradini da 30 s delle rampe).

In [ ]:
LAGS = np.arange(1, 21)


def acf(y):
    return [corr(y[:-k], y[k:]) for k in LAGS]


def segment_residual(r, v):
    changed = np.r_[True, (np.diff(r.rate) != 0) | (np.diff(r.mix) != 0)]
    seg, res = np.cumsum(changed), np.full_like(v, np.nan)
    for i in np.unique(seg):
        on = seg == i
        if on.sum() >= 3:
            res[on] = v[on] - np.nanmedian(v[on], axis=0)
    return res


A, R = {f: [] for f in FEATURES}, {f: [] for f in FEATURES}
for k in clean_ids:
    r = runs[k]
    v = np.where((r.rate >= 0)[:, None, None], Z[k], np.nan)
    res = segment_residual(r, v)
    for j, f in enumerate(FEATURES):
        for s, name in enumerate(SERVICES):
            if status.loc[name, f] == "viva":
                A[f].append(acf(v[:, s, j]))
                R[f].append(acf(res[:, s, j]))
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8), sharey=True)
for ax, data, title in zip(axes, (A, R), ("valori Z", "residuo dentro il segmento di carico")):
    for f in FEATURES:
        if data[f]:
            ax.plot(LAGS * STEP / 60, np.nanmedian(data[f], axis=0), "o-", ms=2, lw=1, label=f)
    ax.axhline(.2, color="k", ls=":")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("lag (minuti)")
axes[0].set_ylabel("autocorrelazione")
axes[1].legend(fontsize=7, ncol=2)
plt.show()
lag_r = {f: LAGS[np.argmax(np.nanmedian(R[f], axis=0) < .2)] * STEP / 60 for f in FEATURES if R[f]}
print("minuti perché il residuo scenda sotto 0.2:", {f: round(m, 1) for f, m in lag_r.items()})
seg_s = pd.Series([s["seconds"] for k in clean_ids for s in runs[k].labels["segments"] if s["seconds"] > 30]) / 60
seg_s.describe().round(1).to_frame("durata segmenti (min)").T

*Quali servizi si muovono insieme?* Correlazione dei pacchetti ricevuti (`Z`) tra servizi, sulle sessioni pulite.
Servizi molto correlati si accendono insieme anche quando il guasto è in uno solo: per la localizzazione sono i
candidati che il modello deve saper separare.

In [ ]:
j = FEATURES.index("rx_packets")
live = [s for s in SERVICES if status.loc[s, "rx_packets"] == "viva"]
rx = pd.DataFrame(np.concatenate([Z[k][runs[k].rate >= 0][:, :, j] for k in clean_ids]), columns=SERVICES)[live]
heat(rx.corr(), "correlazione di rx_packets tra servizi (sessioni pulite)", annot=len(live) <= 18)
plt.show()

**Lettura.** `cpu` è esattamente `cpu_user + cpu_system` (residuo 1e-8 core su una mediana di 5-8 mC): si toglie. Le altre correlazioni alte (rete tra loro 0.99-1.00, CPU con rete 0.93-0.97) restano, perché sono le relazioni che i guasti rompono. Un servizio ha poche dimensioni: k95 = 2-3 per i servizi applicativi di Hotel, 1-4 in Social, e la prima componente (53-95% della varianza) è il carico (|corr| con il rate 0.5-0.86), tranne nei database senza traffico. Per un servizio in un passo bastano 2-4 dimensioni latenti; per una finestra il latente va scelto sul dev, partendo piccolo. Nel tempo i valori restano autocorrelati per circa 4.5 minuti, la durata tipica di un segmento di carico (mediana 5 min, da 2 a 8). Tolto il segmento, CPU e rete sono rumore bianco già a 30 s e la memoria scende sotto 0.2 in 1-2 minuti. La dinamica dentro la finestra è quindi quasi tutta il carico: W = 10 (5 minuti) copre un segmento e i suoi cambi, e finestre più lunghe non aggiungono dinamica propria.

## 6 · Guasti

*Il guasto si vede, e il servizio colpito emerge?* Per ogni run di guasto dello split `SPLIT` lo score di un servizio è
il massimo |z robusto| sulle feature nei `AFTER_S` secondi dopo l'iniezione, con il settle come riferimento
(`explore.score_services`). `rank` = posizione del target (1 = primo). È un rivelatore ingenuo, senza modello: dice quanto
il guasto è visibile nelle sole metriche e dà un'asticella al CMVAE. Il dev è il default; il test va guardato solo per la
valutazione finale.

In [ ]:
AFTER_S = 300
rows = []
for k, r in runs.items():
    if r.split != SPLIT or r.labels["fault_type"] == "No fault":
        continue
    scored = ex.score_services(r, after_s=AFTER_S)
    names = [s for s, *_ in scored]
    targets = ex.target_services(r)
    rank = min((names.index(t) + 1 for t in targets), default=None)
    rows.append({"run": k, "scenario": info.loc[k, "scenario"], "rate_inj": info.loc[k, "rate_inj"],
                 "target": ", ".join(targets), "rank": rank,
                 "score target": scored[rank - 1][1] if rank else np.nan,
                 "primi 3": ", ".join(f"{s} ({f}, {v:.0f})" for s, v, f in scored[:3])})
faults = pd.DataFrame(rows).set_index("run")
display(faults.round(1))
faults.groupby("scenario").agg(run=("rank", "size"), rank_mediano=("rank", "median"),
                               top1=("rank", lambda v: (v == 1).mean()), top3=("rank", lambda v: (v <= 3).mean()))

*Quanto è alto lo stesso score senza guasto?* Lo score massimo sui servizi nelle run senza guasto (sessioni pulite alla
finta iniezione, `noop` di dev e test): è il rumore di fondo. Un target con score sotto questo livello non si distingue da
una run sana con questo rivelatore.

In [ ]:
quiet = pd.DataFrame([dict(zip(["servizio", "score", "feature"], ex.score_services(r, after_s=AFTER_S)[0]), run=k)
                      for k, r in runs.items() if r.labels["fault_type"] == "No fault"]).set_index("run")
print("score massimo senza guasto:", quiet.score.describe()[["count", "50%", "max"]].round(1).to_dict())
for name, q in (("la mediana", quiet.score.median()), ("il massimo", quiet.score.max())):
    print(f"run di guasto con il target sopra {name} senza guasto: {(faults['score target'] > q).mean():.0%}")
display(quiet.sort_values("score").tail(3).round(1))
if SPLIT == "test":
    display(faults.pivot_table(index="scenario", columns="rate_inj", values="rank"))

Per ogni run di guasto (al massimo `MAX_PLOTS`): max |z| sulle feature per servizio nel tempo. Linee colorate = target,
banda grigia = dal 10° al 90° percentile degli altri servizi, tratteggio grigio = il loro massimo.

In [ ]:
MAX_PLOTS = 12
ids = list(faults.index[:MAX_PLOTS])
fig, axes = plt.subplots(-(-len(ids) // 3), 3, figsize=(16, 2.6 * -(-len(ids) // 3)), squeeze=False)
for ax, k in zip(axes.flat, ids):
    r = runs[k]
    zmax = np.nanmax(np.abs(ex.robust_z(r.x, ex.settle_mask(r))), axis=2)
    targets = [SERVICES.index(t) for t in ex.target_services(r)]
    others = np.delete(zmax, targets, axis=1)
    m = r.t_rel / 60
    ax.fill_between(m, np.nanpercentile(others, 10, axis=1), np.nanpercentile(others, 90, axis=1), color=".85")
    ax.plot(m, np.nanmax(others, axis=1), color=".5", ls="--", lw=.8)
    for t in targets:
        ax.plot(m, zmax[:, t], lw=1.5, label=SERVICES[t])
    ax.axvline(0, color="red", ls=":")
    ax.set_yscale("symlog", linthresh=1)
    ax.set_ylim(bottom=0)
    ax.set_title(k.split("/")[1], fontsize=8)
    ax.legend(fontsize=6)
for ax in axes.flat[len(ids):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

**Lettura.** Sul dev il rivelatore ingenuo trova lo stress CPU forte: con stress al 20 e all'80% il target è nei primi 3 in tutte le 8 run delle due app, ed è primo in 6. Fallisce dove il guasto non cambia le metriche del target: net_delay_search in Hotel (rank 8: il ritardo si vede sull'arco, nelle trace), lo stress al 5% in Social (rank 4 e 6), pod_failure di user-timeline-mongodb (rank 2 e 13). Soprattutto non regge il rumore: senza guasto lo score massimo arriva a 455 (Hotel) e 4817 (Social), più di qualunque target. In Hotel sono sessioni pulite in cui il carico sale molto dopo il settle (`rx_bytes` del frontend); in Social è post-storage-memcached, il cui traffico cresce mentre la cache si riempie. Sono i due confondenti visti sopra, carico e stato, ed è questo che il CMVAE deve battere.

## 7 · Spazio libero

Tutto è già caricato: `runs`, `info`, `df`, `P`, `Z`, `status`. Qualche esempio da cui partire.

In [ ]:
# una run, un servizio, alcune feature nel tempo
k = next(k for k in runs if k.startswith("test/"))
df.loc[k].xs(SERVICES[0], level="service")[["cpu_user", "rx_packets", "mem_working_set", "rate"]].plot(subplots=True, figsize=(12, 6))
plt.show()

# mediana per split e fase di una feature, per servizio
df[df.phase != "off"].groupby([df[df.phase != "off"].index.get_level_values("service"), "split", "phase"])["cpu_user"].median().unstack([1, 2]).head()